# PerturbNet on CondOT Sciplex (Start Implementation)

本 notebook 用于在 CondOT 的 sciplex 数据划分与评估标准下复现 PerturbNet，并将结果保存为可直接对比 CondOT/CCOT 的目录结构。

## 1. 安装与导入依赖

- 安装/检查核心依赖
- 导入 PerturbNet 与 condot_repro 工具

In [1]:
# 如当前环境缺失依赖，可取消注释执行
# %pip install -q umap-learn==0.4.6 scvi-tools==0.7.1 scanpy anndata scikit-learn scipy pyarrow

import os
import sys
import shutil
from pathlib import Path
from dataclasses import asdict

# 确保仓库根目录在 sys.path，便于 notebook 直接 import 顶层模块
repo_root = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

import numpy as np
import pandas as pd
import anndata as ad
from scipy import sparse
from tqdm.auto import tqdm
from IPython.display import display

import torch
import scvi

# torch.distributions.Poisson 默认只接受整数采样值；我们的 .X 是 log-normalized 小数。
# 关闭全局 validate_args，让 scvi 的 Poisson 似然能在非整数数据上继续算 log-prob（continuous Poisson 是良定义的）。
torch.distributions.Distribution.set_default_validate_args(False)

if torch.cuda.is_available():
    torch.backends.cudnn.benchmark = True
    if hasattr(torch.backends, "cuda") and hasattr(torch.backends.cuda, "matmul"):
        torch.backends.cuda.matmul.allow_tf32 = True
    if hasattr(torch.backends, "cudnn") and hasattr(torch.backends.cudnn, "allow_tf32"):
        torch.backends.cudnn.allow_tf32 = True
if hasattr(torch, "set_float32_matmul_precision"):
    torch.set_float32_matmul_precision("high")

# PerturbNet
from perturbnet.util import scvi_predictive_z
from perturbnet.cinn.flow import ConditionalFlatCouplingFlow, Net2NetFlow_TFVAE_Covariate_Flow
from perturbnet.cinn.flow_generate import SCVIZ_CheckNet2Net

# CondOT-aligned reproduction helpers (UMAP 留到离线绘制，不在评估循环里导入)
from condot_repro import (
    EvalConfig,
    build_or_validate_condot_split,
    build_split_views,
    summarize_split,
    get_deg_indices,
    compute_metrics,
    ensure_results_dirs,
    save_group_arrays,
    save_stage_metrics,
    save_summary_metrics,
)

print("torch:", torch.__version__)
print("scvi:", scvi.__version__)
print("repo_root:", repo_root)


torch: 1.13.1+cu117
scvi: 0.7.1
repo_root: /home/jamin/Comparision/PerturbNet


## 2. 定义配置与输入参数

In [ ]:
cfg = EvalConfig(
    data_path="/data/jamin_datasets/chemCPA/sciplex_complete_lincs_genes_v3.h5ad",
    output_root="results/perturbnet_condot_sciplex",
    experiment_name="sciplex_comp2condot",
    split_key="split_ood_finetuning",
    condition_key="condition",
    control_key="control",
    cell_type_key="cell_type",
    dose_key="dose",
    smiles_key="SMILES",
    random_seed=42,
    test_size=0.2,
    n_deg=50,
)

# ========== 分支开关 ==========
# 可选: "rdkit2d" 或 "chemcpa"
EMBEDDING_BRANCH = "rdkit2d"
assert EMBEDDING_BRANCH in {"rdkit2d", "chemcpa"}

# 训练相关超参数（按需求预留）
GPU_ID = "0"
USE_CUDA = torch.cuda.is_available()
DEVICE = torch.device(f"cuda:{GPU_ID}" if USE_CUDA else "cpu")
if USE_CUDA:
    torch.cuda.set_device(int(GPU_ID))

SCVI_EPOCHS = 700
SCVI_KL_WARMUP = 400
SCVI_N_LATENT = 10
SCVI_BATCH_SIZE = None
SCVI_NUM_WORKERS = 4

CINN_EPOCHS = 100
CINN_BATCH_SIZE = 128
CINN_LR = 4.5e-6
CINN_NUM_WORKERS = 4
CINN_USE_AMP = True
CINN_AUTO_SAVE_EVERY = 10

MAX_EVAL_CELLS_PER_GROUP = 5000
DO_TRAIN_SCVI = False
DO_TRAIN_CINN = False
# 本次升级把 dose 从 one-hot 改成 continuous scalar，cINN 的 conditioning_dim 会变。
# 如果你的服务器 CINN_MODEL_DIR 下还留有旧版 cINN checkpoint，直接 load 会形状不匹配。
# 首次重训时建议：把 RESUME_IF_AVAILABLE 设为 False 以强制重训，或先把旧 cINN 目录改名备份。
# scVI 的 latent 维度 (SCVI_N_LATENT) 没变，若服务器上 scVI 是用 counts 层训练的可继续复用。
RESUME_IF_AVAILABLE = True
CINN_SAVE_OVERWRITE = True

# .X 是 log-normalized（不是 counts），跳过 counts 层校验，并把 scVI 似然从 nb 改成 normal
REQUIRE_COUNTS_LAYER = False
SCVI_GENE_LIKELIHOOD = "poisson"  # scvi-tools 0.7.1 SCVI 只支持 {nb, zinb, poisson}；log-normalized .X 用 poisson（非整数可接受）

# CondOT 评估按 cell_type + pert + "_1.0" 查 DEG 列表（condot/ccot/evaluate/evaluate.py:208）
EVAL_DOSE_VALUE = 1.0

# embedding 文件直接从本仓库 embeddings/ 目录读取
EMBEDDING_FILE_MAP = {
    "rdkit2d": repo_root / "embeddings" / "rdkit2D_embedding_lincs_trapnell.parquet",
    "chemcpa": repo_root / "embeddings" / "rdkit2D_embedding_lincs_trapnell_chemCPA.parquet",
}

EXPERIMENT_NAME_ACTIVE = f"{cfg.experiment_name}_{EMBEDDING_BRANCH}"
SCVI_MODEL_DIR = Path(cfg.output_root) / EXPERIMENT_NAME_ACTIVE / "models" / "scvi"
CINN_MODEL_DIR = Path(cfg.output_root) / EXPERIMENT_NAME_ACTIVE / "models" / "cinn"
CINN_AUTO_SAVE_DIR = Path(cfg.output_root) / EXPERIMENT_NAME_ACTIVE / "models" / "cinn_autosave"

if hasattr(scvi, "settings") and hasattr(scvi.settings, "seed"):
    scvi.settings.seed = cfg.random_seed
if hasattr(scvi, "settings") and hasattr(scvi.settings, "dl_num_workers"):
    scvi.settings.dl_num_workers = SCVI_NUM_WORKERS

print("DEVICE:", DEVICE)
print("Embedding branch:", EMBEDDING_BRANCH)
print("Active experiment:", EXPERIMENT_NAME_ACTIVE)
print("Config:", asdict(cfg))

In [ ]:
paths = ensure_results_dirs(cfg.output_root, EXPERIMENT_NAME_ACTIVE)
(paths["root"] / "models").mkdir(parents=True, exist_ok=True)

# embedding 文件直接从本仓库 embeddings/ 目录读取
emb_dst = EMBEDDING_FILE_MAP[EMBEDDING_BRANCH]
if not emb_dst.exists():
    raise FileNotFoundError(f"Embedding file not found: {emb_dst}")
print("Embedding file:", emb_dst)

adata = ad.read_h5ad(cfg.data_path)

# === 诊断输出：确认 adata 的 layer / obs 字段，便于判断 scVI 输入与 split 是否可用 ===
print("adata shape        :", adata.shape)
print("adata.layers keys  :", list(adata.layers.keys()))
print("adata.obs columns  :", list(adata.obs.columns))
if cfg.split_key in adata.obs.columns:
    print(f"'{cfg.split_key}' value counts:")
    print(adata.obs[cfg.split_key].value_counts())

required_cols = [cfg.condition_key, cfg.smiles_key, cfg.cell_type_key, cfg.dose_key, cfg.control_key]
missing_cols = [c for c in required_cols if c not in adata.obs.columns]
if missing_cols:
    raise KeyError(f"Missing required obs columns in dataset: {missing_cols}")

# (B1) 严格要求 CondOT 一致的 split 列存在，避免悄悄走 fallback 得到错误划分
assert cfg.split_key in adata.obs.columns, (
    f"Dataset must carry the CondOT-aligned split column '{cfg.split_key}'. "
    f"Found columns: {list(adata.obs.columns)}"
)

# (B2) scVI 必须训练在 counts 层；若缺失直接报错以免静默退化
if REQUIRE_COUNTS_LAYER:
    assert "counts" in adata.layers, (
        "scVI with gene_likelihood='nb' requires raw counts. "
        f"Expected adata.layers['counts']; got layers: {list(adata.layers.keys())}"
    )

# split 列已存在：helper 直接复用
adata = build_or_validate_condot_split(
    adata,
    split_key=cfg.split_key,
    condition_key=cfg.condition_key,
    control_key=cfg.control_key,
    ood_drugs=cfg.ood_drugs,
    random_seed=cfg.random_seed,
    test_size=cfg.test_size,
)

split_summary = summarize_split(
    adata,
    split_key=cfg.split_key,
    condition_key=cfg.condition_key,
    cell_type_key=cfg.cell_type_key,
)
split_summary.to_csv(paths["eval"] / "split_summary.csv", index=False)

split_views = build_split_views(adata, split_key=cfg.split_key, control_key=cfg.control_key)
for k, v in split_views.items():
    print(f"{k}: {v.n_obs} cells, {v.n_vars} genes")

In [4]:
# 构造 CondOT 一致的 train / test / ood 药物集合
train_drugs = sorted(split_views["train_treated"].obs[cfg.condition_key].astype(str).unique())
test_drugs = sorted(split_views["test_treated"].obs[cfg.condition_key].astype(str).unique())
ood_drugs = sorted(split_views["ood"].obs[cfg.condition_key].astype(str).unique())

print("n train drugs:", len(train_drugs))
print("n test drugs:", len(test_drugs))
print("n ood drugs:", len(ood_drugs))
print("ood drugs:", ood_drugs)

n train drugs: 178
n test drugs: 178
n ood drugs: 9
ood drugs: ['Alvespimycin', 'Belinostat', 'Dacinostat', 'Flavopiridol', 'Givinostat', 'Hesperadin', 'Quisinostat', 'TAK-901', 'Tanespimycin']


## 3. 实现核心函数

这里实现 PerturbNet 所需的模型加载、训练/加载分支、以及按药物+细胞类型预测的核心函数。

In [5]:
def to_numpy_dense(x):
    if sparse.issparse(x):
        return x.toarray()
    return np.asarray(x)


def read_condition_embeddings(parquet_path: Path):
    try:
        df = pd.read_parquet(parquet_path)
    except Exception as e:
        raise RuntimeError(
            "Failed to read parquet embedding file. Please install `pyarrow` in the notebook environment."
        ) from e

    candidate_cols = [
        "condition",
        cfg.condition_key,
        "drug",
        "perturbation",
        "condition_name",
        "smiles",
        "canonical_smiles",
    ]
    cond_col = next((c for c in candidate_cols if c in df.columns), None)

    if cond_col is None and (df.index.name is not None or not isinstance(df.index, pd.RangeIndex)):
        idx_name = df.index.name if df.index.name is not None else "index"
        df = df.reset_index().rename(columns={idx_name: "condition"})
        cond_col = "condition"

    if cond_col is None:
        raise KeyError(
            "Embedding parquet must contain a condition identifier column (e.g. `condition`) or use condition names as index."
        )

    if cond_col != "condition":
        df = df.rename(columns={cond_col: "condition"})

    value_cols = [c for c in df.columns if c != "condition"]
    if len(value_cols) == 0:
        raise ValueError("Embedding parquet has no embedding columns.")

    df = df.dropna(subset=["condition"]).copy()
    df["condition"] = df["condition"].astype(str)
    df = df.drop_duplicates(subset=["condition"], keep="first").reset_index(drop=True)
    emb = df[value_cols].to_numpy(dtype=np.float32)
    cond = df["condition"].to_numpy()
    cond2vec = {c: emb[i] for i, c in enumerate(cond)}
    return cond2vec, emb.shape[1], df


# === (B3) 全局 cell_type 映射，train/test/ood 共用，避免 cond_dim 漂移 ===
def build_global_ct_map(adata, cell_type_key):
    cell_types = sorted(adata.obs[cell_type_key].astype(str).unique())
    return {ct: i for i, ct in enumerate(cell_types)}


# === (B4) cov = |ct_map| one-hot + 1 维 continuous dose scalar（对齐 PerturbNet tutorial）===
def build_covariates(df_obs, ct_map):
    n = len(df_obs)
    n_ct = len(ct_map)
    cov = np.zeros((n, n_ct + 1), dtype=np.float32)
    ct_vals = df_obs[cfg.cell_type_key].astype(str).to_numpy()
    # dose 可能是字符串或 Categorical，强制转为 float（失败则抛错）
    dose_vals = pd.to_numeric(df_obs[cfg.dose_key], errors="raise").to_numpy(dtype=np.float32)
    for i in range(n):
        cov[i, ct_map[ct_vals[i]]] = 1.0
    cov[:, n_ct] = dose_vals
    return cov


def encode_conditions(df_obs, cond2vec, condition_key):
    cond_vecs, missing = [], []
    for cond in df_obs[condition_key].astype(str).tolist():
        v = cond2vec.get(cond)
        if v is None:
            missing.append(cond)
            cond_vecs.append(None)
        else:
            cond_vecs.append(v)
    return cond_vecs, sorted(set(missing))


def stack_valid_rows(cond_vecs):
    valid_idx = [i for i, v in enumerate(cond_vecs) if v is not None]
    if len(valid_idx) == 0:
        return np.zeros((0, 1), dtype=np.float32), np.array([], dtype=int)
    mat = np.vstack([cond_vecs[i] for i in valid_idx]).astype(np.float32)
    return mat, np.asarray(valid_idx, dtype=int)


def get_scvi_backbone(model):
    if hasattr(model, "module"):
        return model.module
    if hasattr(model, "model"):
        return model.model
    return model


def has_scvi_checkpoint(model_dir: Path):
    return (model_dir / "model.pt").exists() or (model_dir / "model_params.pt").exists()


def freeze_scvi_model(model):
    backbone = get_scvi_backbone(model)
    if hasattr(backbone, "eval"):
        backbone.eval()
    if hasattr(backbone, "parameters"):
        for param in backbone.parameters():
            param.requires_grad = False
    return model


class CachedScviLatentProvider:
    def __init__(self, latent_matrix):
        self.latent_matrix = np.asarray(latent_matrix, dtype=np.float32)

    def get_latent_representation(self, indices=None, give_mean=False, adata=None):
        if indices is None:
            return self.latent_matrix
        idx = np.asarray(indices, dtype=np.int64)
        return self.latent_matrix[idx]


def train_or_load_scvi(adata_train, do_train=True):
    SCVI_MODEL_DIR.mkdir(parents=True, exist_ok=True)

    if do_train and RESUME_IF_AVAILABLE and has_scvi_checkpoint(SCVI_MODEL_DIR):
        print(f"[INFO] Reusing existing SCVI checkpoint: {SCVI_MODEL_DIR}")
        model = scvi.model.SCVI.load(SCVI_MODEL_DIR, adata=adata_train, use_cuda=USE_CUDA)
    elif do_train:
        model = scvi.model.SCVI(adata_train, n_latent=SCVI_N_LATENT, gene_likelihood=SCVI_GENE_LIKELIHOOD)
        if USE_CUDA and hasattr(model, "to"):
            model = model.to(DEVICE)
        train_kwargs = dict(
            n_epochs=SCVI_EPOCHS,
            train_size=1.0,
            n_epochs_kl_warmup=SCVI_KL_WARMUP,
            frequency=20,
            data_loader_kwargs={"num_workers": SCVI_NUM_WORKERS, "pin_memory": USE_CUDA},
        )
        if SCVI_BATCH_SIZE is not None:
            train_kwargs["batch_size"] = SCVI_BATCH_SIZE
        model.train(**train_kwargs)
        model.save(SCVI_MODEL_DIR, overwrite=True)
    else:
        if not has_scvi_checkpoint(SCVI_MODEL_DIR):
            raise FileNotFoundError(f"SCVI checkpoint not found under: {SCVI_MODEL_DIR}")
        model = scvi.model.SCVI.load(SCVI_MODEL_DIR, adata=adata_train, use_cuda=USE_CUDA)

    if USE_CUDA and hasattr(model, "to"):
        model = model.to(DEVICE)
    model = freeze_scvi_model(model)
    return model


def train_or_load_cinn(
    first_stage_data,
    cond_stage_data,
    cond_dim,
    covariates,
    scvi_model,
    do_train=True,
):
    # 构造并（可选）训练 cINN。
    # 注意：perturb=True + cov_concat="Pert" 下，model_con / perturbToOnehotLib / oneHotData
    # 不会被访问 (perturbnet/cinn/flow.py:3230)。rdkit2D 嵌入直接通过 cond_stage_data 参与，
    # Identity + 空 lib + zeros((1,1)) 是合法占位。
    CINN_MODEL_DIR.mkdir(parents=True, exist_ok=True)

    if USE_CUDA:
        torch.cuda.set_device(int(GPU_ID))

    flow = ConditionalFlatCouplingFlow(
        in_channels=first_stage_data.shape[1],
        conditioning_dim=cond_dim,
        embedding_dim=first_stage_data.shape[1],
        hidden_dim=1024,
        hidden_depth=2,
        n_flows=20,
        conditioning_depth=2,
        activation="none",
        conditioner_use_bn=True,
    )

    model = Net2NetFlow_TFVAE_Covariate_Flow(
        configured_flow=flow,
        first_stage_data=first_stage_data,
        cond_stage_data=cond_stage_data,
        model_con=torch.nn.Identity(),
        std_model=None,
        perturbToOnehotLib={},
        oneHotData=np.zeros((1, 1), dtype=np.float32),
        covariates=covariates,
        scvi_model=scvi_model,
    )
    if hasattr(model, "to"):
        model = model.to(DEVICE)
    print(f"[INFO] cINN device: {next(model.flow.parameters()).device}")

    ckpt_file = CINN_MODEL_DIR / "model_params.pt"

    if do_train and RESUME_IF_AVAILABLE and ckpt_file.exists():
        print(f"[INFO] Reusing existing cINN checkpoint: {ckpt_file}")
        model.load(str(CINN_MODEL_DIR))
        return model

    if not do_train:
        if not ckpt_file.exists():
            raise FileNotFoundError(f"cINN checkpoint not found: {ckpt_file}")
        model.load(str(CINN_MODEL_DIR))
        return model

    model.train_cinn_hgraph(
        n_epochs=CINN_EPOCHS,
        batch_size=CINN_BATCH_SIZE,
        lr=CINN_LR,
        seed=cfg.random_seed,
        train_ratio=0.95,
        cov_concat="Pert",
        perturb=True,
        auto_save=CINN_AUTO_SAVE_EVERY,
        auto_save_path=str(CINN_AUTO_SAVE_DIR),
        num_workers=CINN_NUM_WORKERS,
        pin_memory=USE_CUDA,
    )
    model.save(str(CINN_MODEL_DIR), overwrite=CINN_SAVE_OVERWRITE)
    print(f"[INFO] Saved cINN checkpoint to: {CINN_MODEL_DIR} (overwrite={CINN_SAVE_OVERWRITE})")
    return model


def sample_prediction_for_group(
    generator,
    group_adata,
    cond_vec,
    covariates_group,
    max_cells=None,
):
    # 采样 (drug × cell_type) 下的预测细胞。
    # 返回：(real_counts, pred_counts, lib_sizes)
    # - real_counts 从 layers["counts"] 取，与 scVI 训练输入保持一致
    # - lib_sizes 用每个真实 cell 的 counts 之和（B7：不再硬编码 10000）
    n_cells = group_adata.n_obs
    if max_cells is not None and n_cells > max_cells:
        rs = np.random.RandomState(cfg.random_seed)
        keep = rs.choice(n_cells, size=max_cells, replace=False)
        group_adata = group_adata[keep].copy()
        covariates_group = covariates_group[keep]
        n_cells = group_adata.n_obs

    if "counts" in group_adata.layers:
        real_counts = to_numpy_dense(group_adata.layers["counts"]).astype(np.float32)
    else:
        real_counts = to_numpy_dense(group_adata.X).astype(np.float32)

    # scVI decoder 内部做 px_rate = exp(library) * px_scale：
    # - shape 必须是 (n_cells, 1)，否则与 (n_cells, n_genes) 的 px_scale broadcast 失败
    # - value 是 log 库大小（decoder 里套 exp）
    row_sums = real_counts.sum(axis=1).astype(np.float64)
    row_sums = np.where(row_sums > 0, row_sums, 1.0)  # 避免 log(0)
    log_lib = np.log(row_sums).astype(np.float32).reshape(-1, 1)

    cond_embed = np.tile(cond_vec.astype(np.float32), (n_cells, 1))
    cond_input = np.concatenate([cond_embed, covariates_group.astype(np.float32)], axis=1)

    # SCVIZ_CheckNet2Net.sample_data 返回 (sampled_z, sampled_counts)
    _, sampled_counts = generator.sample_data(
        condition_data=cond_input,
        library_data=log_lib,
        batch_size=min(1024, max(1, n_cells)),
    )
    pred_counts = np.asarray(sampled_counts, dtype=np.float32)
    if pred_counts.ndim == 3 and pred_counts.shape[0] == 1:
        pred_counts = pred_counts[0]
    return real_counts, pred_counts, log_lib


def build_control_views_by_cell_type(control_adata, cell_type_key):
    # 把一段 control adata 按 cell_type 切片，用于评估时提供 source。
    return {
        ct: control_adata[control_adata.obs[cell_type_key].astype(str) == ct].copy()
        for ct in sorted(control_adata.obs[cell_type_key].astype(str).unique())
    }


def sample_control_source_counts(ctrl_view, seed):
    # 返回全部 control 细胞计数矩阵，不做大小截断（调用方用 condot_align_sizes 对齐）。
    if ctrl_view is None or ctrl_view.n_obs == 0:
        return None
    if "counts" in ctrl_view.layers:
        return to_numpy_dense(ctrl_view.layers["counts"]).astype(np.float32)
    return to_numpy_dense(ctrl_view.X).astype(np.float32)


def condot_align_sizes(real, pred, source, cap=5000, seed=42):
    """CondOT 风格 size 对齐。
    1. 若 real 和 source 均超过 cap，各自随机下采样到 cap。
    2. 把较大的一方下采样到较小一方的大小。
    real 与 pred 始终保持配对（同行对应同一细胞）。
    """
    rs = np.random.RandomState(seed)
    n_src = source.shape[0]
    n_tgt = real.shape[0]

    if n_src > cap and n_tgt > cap:
        src_idx = rs.choice(n_src, size=cap, replace=False)
        tgt_idx = rs.choice(n_tgt, size=cap, replace=False)
        source = source[src_idx]
        real = real[tgt_idx]
        pred = pred[tgt_idx]
        n_src, n_tgt = cap, cap

    if n_src > n_tgt:
        idx = rs.choice(n_src, size=n_tgt, replace=False)
        source = source[idx]
    elif n_tgt > n_src:
        idx = rs.choice(n_tgt, size=n_src, replace=False)
        real = real[idx]
        pred = pred[idx]

    return real, pred, source

In [6]:
# 只用 train split 训练
adata_train = split_views["train_treated"].copy()
adata_control_train = split_views["train_control"].copy()

# 读取分支 embedding
condition_to_embedding, embedding_dim, emb_df = read_condition_embeddings(emb_dst)
emb_df.to_csv(paths["eval"] / f"embedding_table_{EMBEDDING_BRANCH}.csv", index=False)

# 全局 cell_type map（B3），train/test/ood 共用 → cond_dim 不会在评估时漂移
global_ct_map = build_global_ct_map(adata, cfg.cell_type_key)
print("Global cell_type map:", global_ct_map)

cov_train = build_covariates(adata_train.obs, global_ct_map)
cond_vecs_train, missing_train = encode_conditions(
    adata_train.obs,
    condition_to_embedding,
    condition_key=cfg.smiles_key,
)
if missing_train:
    pd.DataFrame({"missing_condition": missing_train}).to_csv(
        paths["eval"] / f"missing_conditions_in_{EMBEDDING_BRANCH}.csv", index=False
    )

cond_embed_train, valid_idx = stack_valid_rows(cond_vecs_train)
if len(valid_idx) < adata_train.n_obs:
    adata_train = adata_train[valid_idx].copy()
cov_train_valid = cov_train[valid_idx].astype(np.float32)

cond_dim_total = cond_embed_train.shape[1] + cov_train_valid.shape[1]
print(
    f"cond_dim_total={cond_dim_total} "
    f"(embedding={cond_embed_train.shape[1]}, covariates={cov_train_valid.shape[1]})"
)

# SCVI 输入：若 adata 有 counts 层就用它，否则退回 .X（需配合 SCVI_GENE_LIKELIHOOD）
if "counts" in adata_train.layers:
    scvi.data.setup_anndata(adata_train, layer="counts")
else:
    scvi.data.setup_anndata(adata_train)

model_scvi = train_or_load_scvi(adata_train, do_train=DO_TRAIN_SCVI)

# scVI latent 导出
latent_train = model_scvi.get_latent_representation(adata_train, give_mean=False)
latent_train = np.asarray(latent_train, dtype=np.float32)
print("latent_train shape:", latent_train.shape)
np.save(paths["eval"] / f"latent_train_{EMBEDDING_BRANCH}.npy", latent_train)

# cINN
scvi_latent_provider = CachedScviLatentProvider(latent_train)
model_cinn = train_or_load_cinn(
    first_stage_data=latent_train,
    cond_stage_data=cond_embed_train,
    cond_dim=cond_dim_total,
    covariates=cov_train_valid,
    scvi_model=scvi_latent_provider,
    do_train=DO_TRAIN_CINN,
)
model_cinn.flow.eval()

# 用独立 CPU 版 SCVI checkpoint 构造解码器，避免对象复用导致设备污染
if has_scvi_checkpoint(SCVI_MODEL_DIR):
    model_scvi_for_decode = scvi.model.SCVI.load(SCVI_MODEL_DIR, adata=adata_train, use_cuda=False)
else:
    model_scvi_for_decode = model_scvi
if hasattr(model_scvi_for_decode, "to"):
    model_scvi_for_decode = model_scvi_for_decode.to("cpu")
scvi_decoder = scvi_predictive_z(model_scvi_for_decode)
scviz = SCVIZ_CheckNet2Net(model_cinn, DEVICE, scvi_decoder)
print("Models ready.")


Global cell_type map: {'A549': 0, 'K562': 1, 'MCF7': 2}
cond_dim_total=198 (embedding=194, covariates=4)
INFO     No batch_key inputted, assuming all cells are same batch                                                  
INFO     No label_key inputted, assuming all cells have same label                                                 
INFO     Using data from adata.X                                                                                   
INFO     Computing library size prior per batch                                                                    


/data/jamin/conda_envs/PerturbNet/lib/python3.7/site-packages/scvi/data/_anndata.py:760: UserWarning: adata.X does not contain unnormalized count data. Are you sure this is what you want?


WARNING  This dataset has some empty cells, this might fail inference.Data should be filtered with                 
         `scanpy.pp.filter_cells()`                                                                                
INFO     Successfully registered anndata object containing 526369 cells, 977 vars, 1 batches, 1 labels, and 0      
         proteins. Also registered 0 extra categorical covariates and 0 extra continuous covariates.               
INFO     Please do not further modify adata until model is trained.                                                
INFO     Using data from adata.X                                                                                   
INFO     Computing library size prior per batch                                                                    
WARNING  This dataset has some empty cells, this might fail inference.Data should be filtered with                 
         `scanpy.pp.filter_cells()`                                     

## 4. 编写主执行流程

按 CondOT 风格分别在 test / ood 评估，并保存：
- 每个药物 + 细胞类型的 real/pred/source 数组
- 每个 stage 的详细指标
- stage summary（mean/std）
- UMAP 图

In [7]:
def evaluate_stage(stage_name, adata_stage, control_views, generator):
    """按 CondOT 约定做 per-(drug × cell_type) 双层循环评估。
    对每组同时计算：
    - full 空间 (1000 维)
    - DEG 50 子空间
    并分别保存到 data/<stage>_full/<pert>/<cell_type>.npz 和 data/<stage>_deg50/...

    UMAP 按 condot 方式离线绘制，这里只落盘数据与指标。
    """
    rows_full, rows_deg50, skipped = [], [], []
    cov_stage = build_covariates(adata_stage.obs, global_ct_map)

    perts = sorted(adata_stage.obs[cfg.condition_key].astype(str).unique())
    cell_types_in_stage = sorted(adata_stage.obs[cfg.cell_type_key].astype(str).unique())

    for pert in tqdm(perts, desc=f"{stage_name} perts"):
        for ct in cell_types_in_stage:
            mask = (
                (adata_stage.obs[cfg.condition_key].astype(str) == pert)
                & (adata_stage.obs[cfg.cell_type_key].astype(str) == ct)
            )
            n_sel = int(mask.sum())
            if n_sel < 5:
                skipped.append({"stage": stage_name, "pert": pert, "cell_type": ct, "reason": f"n={n_sel}<5"})
                continue

            adata_grp = adata_stage[mask].copy()

            smiles_vals = adata_grp.obs[cfg.smiles_key].astype(str).unique()
            if len(smiles_vals) != 1:
                skipped.append({"stage": stage_name, "pert": pert, "cell_type": ct, "reason": f"non-unique SMILES ({len(smiles_vals)})"})
                continue
            smiles = smiles_vals[0]
            cond_vec = condition_to_embedding.get(smiles)
            if cond_vec is None:
                skipped.append({"stage": stage_name, "pert": pert, "cell_type": ct, "reason": f"SMILES missing in {EMBEDDING_BRANCH} embedding: {smiles}"})
                continue

            grp_idx = np.where(mask.to_numpy())[0]
            cov_group = cov_stage[grp_idx]

            try:
                real_counts, pred_counts, _lib = sample_prediction_for_group(
                    generator,
                    adata_grp,
                    cond_vec=cond_vec,
                    covariates_group=cov_group,
                    max_cells=MAX_EVAL_CELLS_PER_GROUP,
                )
            except Exception as e:
                skipped.append({"stage": stage_name, "pert": pert, "cell_type": ct, "reason": f"sample_data failed: {e}"})
                continue

            # source：对应 cell_type 的全部 control 细胞（CondOT 约定）
            src_counts = sample_control_source_counts(
                control_views.get(ct),
                seed=cfg.random_seed,
            )
            if src_counts is None:
                src_counts = np.zeros_like(pred_counts)

            # CondOT 风格大小对齐：当 real > source 时下采样 real+pred，而非上采样 source
            real_counts, pred_counts, src_counts = condot_align_sizes(
                real_counts, pred_counts, src_counts,
                cap=MAX_EVAL_CELLS_PER_GROUP,
                seed=cfg.random_seed,
            )

            # ===== full 空间 =====
            m_full = compute_metrics(real_counts, pred_counts)
            m_full.update({"stage": stage_name, "condition": pert, "cell_type": ct, "n_cells": int(real_counts.shape[0])})
            rows_full.append(m_full)
            save_group_arrays(
                data_dir=paths["data"],
                stage=stage_name,
                condition=pert,
                cell_type=ct,
                real=real_counts,
                pred=pred_counts,
                source=src_counts,
                subset_name="full",
            )

            # ===== DEG 50 子空间 =====
            try:
                deg_idx = get_deg_indices(
                    adata,
                    condition=pert,
                    cell_type=ct,
                    dose_value=EVAL_DOSE_VALUE,
                    n_deg=cfg.n_deg,
                )
            except (KeyError, ValueError) as e:
                skipped.append({"stage": stage_name, "pert": pert, "cell_type": ct, "reason": f"DEG lookup failed: {e}"})
                continue

            m_deg = compute_metrics(real_counts[:, deg_idx], pred_counts[:, deg_idx])
            m_deg.update({"stage": stage_name, "condition": pert, "cell_type": ct, "n_cells": int(real_counts.shape[0])})
            rows_deg50.append(m_deg)
            save_group_arrays(
                data_dir=paths["data"],
                stage=stage_name,
                condition=pert,
                cell_type=ct,
                real=real_counts[:, deg_idx],
                pred=pred_counts[:, deg_idx],
                source=src_counts[:, deg_idx] if src_counts.shape[1] == real_counts.shape[1] else np.zeros_like(pred_counts[:, deg_idx]),
                subset_name="deg50",
            )

    df_full = pd.DataFrame(rows_full)
    df_deg50 = pd.DataFrame(rows_deg50)
    if not df_full.empty:
        save_stage_metrics(eval_dir=paths["eval"], stage=stage_name, stage_metrics=df_full, subset_name="full")
    if not df_deg50.empty:
        save_stage_metrics(eval_dir=paths["eval"], stage=stage_name, stage_metrics=df_deg50, subset_name="deg50")
    if skipped:
        pd.DataFrame(skipped).to_csv(paths["eval"] / f"skipped_{stage_name}.csv", index=False)

    return {"full": df_full, "deg50": df_deg50, "skipped": skipped}

In [ ]:
# 按 CondOT 约定：test 用 test_control、ood 也用 test_control 作为 source cells
test_control_views = build_control_views_by_cell_type(split_views["test_control"], cfg.cell_type_key)
ood_control_views = test_control_views

stage_results = {
    "test": evaluate_stage("test", split_views["test_treated"], test_control_views, scviz),
    "ood": evaluate_stage("ood", split_views["ood"], ood_control_views, scviz),
}

# summary：full / deg50 各一张，内含 (stage × {mean,std}) 四行
for subset in ("full", "deg50"):
    rows = []
    for name, bundle in stage_results.items():
        df = bundle[subset]
        if df.empty:
            continue
        metric_cols = [c for c in ["mmd", "energy_distance", "r2", "l2_loss", "fid"] if c in df.columns]
        rows.append({"stage": name, "stat": "mean", **df[metric_cols].mean().to_dict()})
        rows.append({"stage": name, "stat": "std", **df[metric_cols].std().to_dict()})
    summary_df = pd.DataFrame(rows)
    save_summary_metrics(eval_dir=paths["eval"], summary_df=summary_df, subset_name=subset)
    print(f"--- summary_{subset}_results ---")
    display(summary_df)

In [9]:
# 设备一致性快速检查
if "scviz" in globals():
    try:
        flow_dev = next(scviz.model.flow.parameters()).device
        print("flow device:", flow_dev)
    except Exception as e:
        print("flow device check failed:", e)
    try:
        dec_dev = next(scviz.scvi_model_decode.model.model.decoder.parameters()).device
        print("decoder device:", dec_dev)
    except Exception as e:
        print("decoder device check failed:", e)
else:
    print("scviz not built yet. Run the training/evaluation cells first.")


flow device: cuda:0
decoder device: cpu


## 5. 添加单元测试用例

对关键工具函数做最小可运行测试（正常路径 + 边界路径）。

In [ ]:
def _test_compute_metrics_shape_guard():
    a = np.random.randn(10, 5)
    b = np.random.randn(12, 5)
    try:
        _ = compute_metrics(a, b)
        raise AssertionError("Expected shape mismatch error")
    except ValueError:
        pass


def _test_compute_metrics_basic():
    a = np.random.randn(20, 8)
    out = compute_metrics(a, a.copy())
    assert set(["mmd", "energy_distance", "r2", "l2_loss", "fid"]).issubset(out.keys())


def _test_split_summary_not_empty():
    s = summarize_split(
        adata,
        split_key=cfg.split_key,
        condition_key=cfg.condition_key,
        cell_type_key=cfg.cell_type_key,
    )
    assert s.shape[0] > 0


def _test_global_cov_shape():
    cov = build_covariates(adata_train.obs.head(3), global_ct_map)
    assert cov.shape[1] == len(global_ct_map) + 1, f"cov dim unexpected: {cov.shape}"


_test_compute_metrics_shape_guard()
_test_compute_metrics_basic()
_test_split_summary_not_empty()
_test_global_cov_shape()
print("unit tests passed")

## 6. 运行与输出验证

检查输出目录、关键文件与结果表头，快速确认实验流程已完整落盘。

In [11]:
print("Result root:", paths["root"])
print("Eval dir:", paths["eval"])
print("Data dir:", paths["data"])
print("Plot dir:", paths["plot"], "(UMAP 离线绘制)")

expected_files = [
    paths["eval"] / "split_summary.csv",
    paths["eval"] / "test_full_results.csv",
    paths["eval"] / "test_deg50_results.csv",
    paths["eval"] / "ood_full_results.csv",
    paths["eval"] / "ood_deg50_results.csv",
    paths["eval"] / "summary_full_results.csv",
    paths["eval"] / "summary_deg50_results.csv",
]

for p in expected_files:
    print(p, "exists:", p.exists())

for subset in ("full", "deg50"):
    p = paths["eval"] / f"summary_{subset}_results.csv"
    if p.exists():
        print(f"\n--- summary_{subset} ---")
        display(pd.read_csv(p).head())


Result root: results/perturbnet_condot_sciplex/sciplex_comp2condot_rdkit2d
Eval dir: results/perturbnet_condot_sciplex/sciplex_comp2condot_rdkit2d/eval
Data dir: results/perturbnet_condot_sciplex/sciplex_comp2condot_rdkit2d/eval/data
Plot dir: results/perturbnet_condot_sciplex/sciplex_comp2condot_rdkit2d/eval/plot (UMAP 离线绘制)
results/perturbnet_condot_sciplex/sciplex_comp2condot_rdkit2d/eval/split_summary.csv exists: True
results/perturbnet_condot_sciplex/sciplex_comp2condot_rdkit2d/eval/test_full_results.csv exists: True
results/perturbnet_condot_sciplex/sciplex_comp2condot_rdkit2d/eval/test_deg50_results.csv exists: True
results/perturbnet_condot_sciplex/sciplex_comp2condot_rdkit2d/eval/ood_full_results.csv exists: True
results/perturbnet_condot_sciplex/sciplex_comp2condot_rdkit2d/eval/ood_deg50_results.csv exists: True
results/perturbnet_condot_sciplex/sciplex_comp2condot_rdkit2d/eval/summary_full_results.csv exists: True
results/perturbnet_condot_sciplex/sciplex_comp2condot_rdkit2d

,Unnamed: 0,stage,stat,mmd,E_d,R2,L2,FID
0,0,test,mean,0.056751,0.186923,0.709121,1.802665,88.146251
1,1,test,std,0.021346,0.195557,0.165521,0.392797,6.680416
2,2,ood,mean,0.012582,0.248917,0.890598,0.994495,42.368478
3,3,ood,std,0.005278,0.210058,0.107040,0.419364,5.003861



--- summary_deg50 ---


,Unnamed: 0,stage,stat,mmd,E_d,R2,L2,FID
0,0,test,mean,0.047039,0.097214,0.737425,0.543286,3.060486
1,1,test,std,0.017636,0.103715,0.260588,0.197871,1.622795
2,2,ood,mean,0.021418,0.234767,0.761529,0.633773,2.427348
3,3,ood,std,0.012993,0.259050,0.309629,0.434517,2.147816
